# Cross-Dataset Baseline

The mixed-source baseline is a useful signal check, but it may benefit from dataset-specific sensor properties. This notebook trains the same participant-weighted summary-feature classifier on one main dataset and tests it on the other.

This is the stricter generalization test that must be passed before interpreting a CNN as learning stroke gait rather than source identity.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, f1_score, roc_auc_score
from sklearn.preprocessing import StandardScaler

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
PROCESSED = PROJECT_ROOT / 'data' / 'processed'
windows = np.load(PROCESSED / 'validated_gait_windows_float32.npy', mmap_mode='r')
metadata = pd.read_csv(PROCESSED / 'validated_window_metadata.csv')
metadata['label_binary'] = metadata['label'].map({'healthy': 0, 'stroke': 1}).astype(int)
print('Windows:', windows.shape)
print(metadata.groupby(['dataset_id', 'label']).size())


Windows: (18511, 500, 18)
dataset_id    label  
felius_2024   healthy     2921
              stroke     13445
voisard_2025  healthy     1039
              stroke      1106
dtype: int64


In [2]:
def summary_features(window_array, batch_size=1024):
    chunks = []
    for start in range(0, len(window_array), batch_size):
        batch = np.asarray(window_array[start:start + batch_size], dtype=np.float32)
        chunks.append(np.concatenate([batch.mean(axis=1), batch.std(axis=1), np.sqrt(np.mean(batch ** 2, axis=1))], axis=1))
    return np.concatenate(chunks, axis=0)

X = summary_features(windows)
assert np.isfinite(X).all()

results = []
predictions = []
for train_dataset, test_dataset in [('voisard_2025', 'felius_2024'), ('felius_2024', 'voisard_2025')]:
    train_mask = metadata['dataset_id'].eq(train_dataset)
    test_mask = metadata['dataset_id'].eq(test_dataset)
    train_indices = metadata.index[train_mask].to_numpy()
    test_indices = metadata.index[test_mask].to_numpy()
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X[train_indices])
    X_test = scaler.transform(X[test_indices])
    y_train = metadata.loc[train_mask, 'label_binary'].to_numpy()
    y_test = metadata.loc[test_mask, 'label_binary'].to_numpy()
    counts = metadata.loc[train_mask].groupby('participant_key').size()
    weights = metadata.loc[train_mask, 'participant_key'].map(1.0 / counts).to_numpy()
    weights = weights / weights.mean()
    model = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
    model.fit(X_train, y_train, sample_weight=weights)
    probabilities = model.predict_proba(X_test)[:, 1]
    test_frame = metadata.loc[test_mask, ['participant_key', 'label', 'label_binary']].copy()
    test_frame['probability'] = probabilities
    participant_frame = test_frame.groupby(['participant_key', 'label', 'label_binary'], as_index=False)['probability'].mean()
    y_true = participant_frame['label_binary'].to_numpy()
    y_prob = participant_frame['probability'].to_numpy()
    results.append({
        'train_dataset': train_dataset,
        'test_dataset': test_dataset,
        'test_participants': len(participant_frame),
        'balanced_accuracy': balanced_accuracy_score(y_true, (y_prob >= 0.5).astype(int)),
        'roc_auc': roc_auc_score(y_true, y_prob),
        'f1': f1_score(y_true, (y_prob >= 0.5).astype(int)),
    })
    participant_frame['train_dataset'] = train_dataset
    participant_frame['test_dataset'] = test_dataset
    predictions.append(participant_frame)

results = pd.DataFrame(results)
predictions = pd.concat(predictions, ignore_index=True)
print(results.round(3).to_string(index=False))


train_dataset test_dataset  test_participants  balanced_accuracy  roc_auc    f1
 voisard_2025  felius_2024                163               0.59    0.837 0.344
  felius_2024 voisard_2025                121               0.50    0.148 0.000


In [3]:
results.to_csv(PROCESSED / 'cross_dataset_baseline_results.csv', index=False)
predictions.to_csv(PROCESSED / 'cross_dataset_baseline_predictions.csv', index=False)
print('Cross-dataset outputs written to data/processed/')


Cross-dataset outputs written to data/processed/


## Decision gate

If both directions retain useful ROC-AUC and balanced accuracy, the common signal representation is a credible starting point for a CNN. If performance collapses in one or both directions, the CNN must include stronger harmonization controls and cross-dataset results should be treated as the primary evaluation.